# Option A — component test (stages 3→6) on curated real hotspots

Runs the scientifically interesting back half of the neoantigen pipeline
— candidate windows → **MHCflurry** presentation gate → **Łuksza** recognition
composite → string-of-beads construct — on a curated panel of **real** oncogenic
hotspot mutations (KRAS G12D/G12V, BRAF V600E, TP53 R175H, PIK3CA H1047R, EGFR L858R).

**Honesty:** these are real mutations on real human proteins, but this notebook makes
no claim any tumour carries them — it proves the *ranking science runs on real
mutations*, in isolation from the genomics front end. The full-tumour run is Option B.
See `docs/e2e_validation_notes.md`.

_First-run note: expect to iterate once (MHCflurry allele support, proteome download,
package paths). That's the point of the liveness run._

In [ ]:
# 1. Get the repo onto Colab and on sys.path (it must contain core.py at its root).
#    Option A: mount Drive if the repo lives there.
# from google.colab import drive; drive.mount('/content/drive')
# REPO = '/content/drive/MyDrive/neoantigen_pipeline'
#    Option B: clone the private repo (needs a GitHub token with repo scope).
# !git clone https://<TOKEN>@github.com/jclevitt1/neoantigen-pipeline.git /content/neoantigen_pipeline
REPO = '/content/neoantigen_pipeline'
import sys; sys.path.insert(0, REPO)
print('repo on path:', REPO)

In [ ]:
# 2. Install MHCflurry + fetch the presentation models (offline, CPU-fine).
!pip -q install mhcflurry
!mhcflurry-downloads fetch models_class1_presentation

In [ ]:
# 3. A real human proteome for the Łuksza dissimilarity-to-self term
#    (Swiss-Prot human, ~20k proteins — small).
!wget -qO /content/human.fasta.gz 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&compressed=true&query=organism_id:9606+AND+reviewed:true'
!gunzip -f /content/human.fasta.gz
!grep -c '^>' /content/human.fasta  # sanity: number of proteins

In [ ]:
# 4. Assemble + run stages 2a(fixture) → 2b(known HLA) → 3 → 4 → 5 → 6.
from NeoantigenVaccineConstructionPipeline.demos.component_test import run_component_test
out = run_component_test(workdir='/content/run_A', proteome='/content/human.fasta')
out

In [ ]:
# 5. Inspect the outputs: the ranked neoepitopes and the vaccine construct.
import pandas as pd
print('=== RANKED (top of stage 4) ===')
display(pd.read_csv(out['ranked_tsv'], sep='\t').head(20))
print('=== FILTERED (stage 5 survivors) ===')
display(pd.read_csv(out['filtered_tsv'], sep='\t'))
print('=== CONSTRUCT (stage 6 string-of-beads) ===')
print(out['construct_fasta'].read_text())